In [ ]:
import pandas as pd

# --- Load PitchBook deals file ---
file_path = r"D:\vc-research\all deals together.xlsx"
df = pd.read_excel(file_path)

# --- Standardize the company field ---
# Ensure consistent string formatting
df['Companies'] = (
    df['Companies']
    .astype(str)
    .str.strip()
    .str.lower()
)

# --- Drop duplicates based on Companies field ---
df_unique = df.drop_duplicates(subset=['Companies']).copy()

# --- Assign unique IDs ---
df_unique = df_unique.reset_index(drop=True)
df_unique['company_uid'] = df_unique.index + 1  # starts from 1

# Show result
print(df_unique[['Companies', 'company_uid']].head())

# --- OPTIONAL: Save cleaned unique mapping ---
output_path = r"D:\vc-research\pitchbook_company_ids.csv"
df_unique[['Companies', 'company_uid']].to_csv(output_path, index=False)

print(f"Unique company ID file saved to:\n{output_path}")


In [6]:
import re
import pandas as pd
from rapidfuzz import process, fuzz

# ---------------------------------------------------
# 1. File paths (EDIT THESE AS NEEDED)
# ---------------------------------------------------
pitchbook_file = r"D:\vc-research\all deals together.csv"
vc_coi_file    = r"D:\vc-research\vc-research\Reese's contributions\reese data\downloaded_batches\fullCOISample.csv"

# Column in your VC COI dataset that holds company names
VC_COMPANY_COL = "company_name"   # e.g., "issuer_name", etc.

# NEW: stable VC identifier column to carry through outputs
VC_ID_COL = "company_id"

# Fuzzy match cutoff – adjust as you like
SCORE_THRESHOLD = 90

# ---------------------------------------------------
# 2. Helper: normalize / clean company names
# ---------------------------------------------------
LEGAL_SUFFIXES = [
    "inc", "inc.", "llc", "corp", "corp.", "co", "co.", "ltd", "ltd.",
    "incorporated", "company", "corporation", "plc", "gmbh", "s.a.", "s.a", "sa"
]

def normalize_company_name(name: str) -> str:
    """
    Normalize company name for fuzzy matching:
    - lowercase
    - remove punctuation
    - remove legal suffixes at end
    - collapse multiple spaces
    """
    if not isinstance(name, str):
        return ""
    
    # Lowercase
    s = name.lower().strip()

    # Replace punctuation with space
    s = re.sub(r"[^a-z0-9]+", " ", s)

    # Remove legal suffixes at the end
    tokens = s.split()
    while tokens and tokens[-1] in LEGAL_SUFFIXES:
        tokens.pop()
    s = " ".join(tokens)

    # Collapse multiple spaces
    s = re.sub(r"\s+", " ", s).strip()

    return s

# ---------------------------------------------------
# 3. Load data
# ---------------------------------------------------
pb_df = pd.read_csv(pitchbook_file)
vc_df = pd.read_csv(vc_coi_file)

# Basic sanity checks
if "Companies" not in pb_df.columns:
    raise ValueError("PitchBook file must contain a 'Companies' column.")

missing_vc_cols = [c for c in [VC_COMPANY_COL, VC_ID_COL] if c not in vc_df.columns]
if missing_vc_cols:
    raise ValueError(f"VC COI file must contain these columns: {missing_vc_cols}")

# ---------------------------------------------------
# 4. Clean / normalize company names
# ---------------------------------------------------
pb_df["Companies_raw"] = pb_df["Companies"].astype(str)
pb_df["name_clean"] = pb_df["Companies_raw"].map(normalize_company_name)

vc_df["company_raw"] = vc_df[VC_COMPANY_COL].astype(str)
vc_df["name_clean"] = vc_df["company_raw"].map(normalize_company_name)

# ---------------------------------------------------
# 5. Deduplicate PitchBook companies & assign IDs
# ---------------------------------------------------
pb_unique = (
    pb_df
    .drop_duplicates(subset=["name_clean"])
    .copy()
)

# Assign unique IDs (if you haven't already)
if "company_uid" not in pb_unique.columns:
    pb_unique = pb_unique.reset_index(drop=True)
    pb_unique["company_uid"] = pb_unique.index + 1

# We'll match VC names against these cleaned PitchBook names
pb_choices = dict(pb_unique["name_clean"])  # {pb_index: "name_clean"}

# ---------------------------------------------------
# 6. Fuzzy match VC companies to PitchBook companies (ROW-LEVEL)
#    Key point: keep vc_index for safe 1:1 merge back to vc_df,
#    but ALSO carry company_id through match outputs.
# ---------------------------------------------------
matches = []

count = 0
n_total = len(vc_df)

for idx, row in vc_df.iterrows():
    query = row["name_clean"]
    vc_company_id = row[VC_ID_COL]

    print(f"Processing {count}/{n_total}")
    count += 1

    # Skip empty names
    if not query:
        matches.append({
            "vc_index": idx,
            "company_id": vc_company_id,  # <-- carry VC company_id
            "vc_company_raw": row["company_raw"],
            "vc_company_clean": query,
            "pb_index": None,
            "pb_company_raw": None,
            "pb_company_clean": None,
            "company_uid": None,
            "match_score": 0,
        })
        continue

    # extractOne returns (matched_value, score, key)
    match = process.extractOne(
        query,
        pb_choices,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=0  # we'll filter ourselves
    )

    if match is None:
        matches.append({
            "vc_index": idx,
            "company_id": vc_company_id,  # <-- carry VC company_id
            "vc_company_raw": row["company_raw"],
            "vc_company_clean": query,
            "pb_index": None,
            "pb_company_raw": None,
            "pb_company_clean": None,
            "company_uid": None,
            "match_score": 0,
        })
        continue

    matched_value, score, pb_index = match
    pb_row = pb_unique.loc[pb_index]

    matches.append({
        "vc_index": idx,
        "company_id": vc_company_id,  # <-- carry VC company_id
        "vc_company_raw": row["company_raw"],
        "vc_company_clean": query,
        "pb_index": pb_index,
        "pb_company_raw": pb_row["Companies_raw"],
        "pb_company_clean": pb_row["name_clean"],
        "company_uid": pb_row["company_uid"],
        "match_score": score,
    })

matches_df = pd.DataFrame(matches)

# ---------------------------------------------------
# 7. Split into high-confidence matches and others
# ---------------------------------------------------
high_confidence_matches = matches_df[matches_df["match_score"] >= SCORE_THRESHOLD].copy()
low_confidence_matches  = matches_df[matches_df["match_score"] < SCORE_THRESHOLD].copy()

# ---------------------------------------------------
# 8. Merge IDs back into the VC COI dataframe (SAFE 1:1 via vc_index)
# ---------------------------------------------------
# IMPORTANT: keep merge on vc_index / left_index to avoid many-to-many blowups,
# while still having company_id in outputs for later use.
vc_with_matches = vc_df.merge(
    high_confidence_matches[["vc_index", "company_id", "company_uid", "match_score", "pb_company_raw"]],
    left_index=True,
    right_on="vc_index",
    how="left"
)

# ---------------------------------------------------
# 9. Save outputs (EDIT PATHS IF YOU LIKE)
# ---------------------------------------------------
output_match_file     = r"D:\vc-research\vc_coi_pitchbook_matches.csv"
output_low_conf_file  = r"D:\vc-research\vc_coi_pitchbook_matches_lowconfidence.csv"
output_vc_with_ids    = r"D:\vc-research\vc_coi_with_company_uid.csv"

print("High-confidence matches saved to:", output_match_file)
print("Low-confidence / ambiguous matches saved to:", output_low_conf_file)

high_confidence_matches.to_csv(output_match_file, index=False)
low_confidence_matches.to_csv(output_low_conf_file, index=False)

vc_with_matches.to_csv(output_vc_with_ids, index=False)
print("VC COI data with company_uid saved to:", output_vc_with_ids)


C:\Users\Owner\AppData\Local\Temp\ipykernel_92816\3317408901.py:59: DtypeWarning: Columns (3,4,30,38,39,53,58,64,67,69,85,86,87,88,89,90,92,97,98,99,100,101,103,104,105,106,112,113,114,116,117,119,120,122,123,124,125,126,127,128,129,130,131,132,133,144,145,150,151,152,153,155,156,157,162,167,168,169,170,171,174,175,176) have mixed types. Specify dtype option on import or set low_memory=False.
  pb_df = pd.read_csv(pitchbook_file)
C:\Users\Owner\AppData\Local\Temp\ipykernel_92816\3317408901.py:60: DtypeWarning: Columns (7,9,13,15,18,19,20,21,22,23,35,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,16

Processing 0/28616
Processing 1/28616
Processing 2/28616
Processing 3/28616
Processing 4/28616
Processing 5/28616
Processing 6/28616
Processing 7/28616
Processing 8/28616
Processing 9/28616
Processing 10/28616
Processing 11/28616
Processing 12/28616
Processing 13/28616
Processing 14/28616
Processing 15/28616
Processing 16/28616
Processing 17/28616
Processing 18/28616
Processing 19/28616
Processing 20/28616
Processing 21/28616
Processing 22/28616
Processing 23/28616
Processing 24/28616
Processing 25/28616
Processing 26/28616
Processing 27/28616
Processing 28/28616
Processing 29/28616
Processing 30/28616
Processing 31/28616
Processing 32/28616
Processing 33/28616
Processing 34/28616
Processing 35/28616
Processing 36/28616
Processing 37/28616
Processing 38/28616
Processing 39/28616
Processing 40/28616
Processing 41/28616
Processing 42/28616
Processing 43/28616
Processing 44/28616
Processing 45/28616
Processing 46/28616
Processing 47/28616
Processing 48/28616
Processing 49/28616
Processing

In [7]:
import os
import re
import shutil
import pandas as pd

# -----------------------------
# Inputs
# -----------------------------
CSV_PATH = r"D:\vc-research\vc_coi_pitchbook_matches.csv"

BATCH_DIRS = [
    r"D:\vc-research\vc-research\Batch1",
    r"D:\vc-research\vc-research\Batch2",
    r"D:\vc-research\vc-research\Batch3",
    r"D:\vc-research\vc-research\Batch4",
    r"D:\vc-research\vc-research\Batch56",
]

# Where to copy PDFs
OUT_ROOT = r"D:\vc-research\vc-research\matched_pdfs_by_score"
OUT_100  = os.path.join(OUT_ROOT, "match_100")
OUT_LT   = os.path.join(OUT_ROOT, "match_less_than_100")

# Column names in your CSV
ID_COL    = "company_id"
SCORE_COL = "match_score"   # change if your column is named differently

# -----------------------------
# Helpers
# -----------------------------
def safe_makedirs(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def extract_leading_id(filename: str):
    """
    Extract digits before first underscore, e.g.:
    '12345_somefile.pdf' -> '12345'
    Returns None if it doesn't match pattern.
    """
    m = re.match(r"^(\d+)_", filename)
    return m.group(1) if m else None

def normalize_id(x):
    """Coerce IDs to clean digit strings, returns None if missing/invalid."""
    if pd.isna(x):
        return None
    s = str(x).strip()
    # if it's like 12345.0 from CSV reading
    if re.fullmatch(r"\d+\.0", s):
        s = s[:-2]
    # keep only digits if it's purely digits
    if re.fullmatch(r"\d+", s):
        return s
    # if user has something like 'ID: 12345', try to find digits
    m = re.search(r"(\d+)", s)
    return m.group(1) if m else None

def is_hundred(score: float) -> bool:
    """Treat 100 (0-100 scale) or 1.0 (0-1 scale) as a perfect match."""
    if score is None or pd.isna(score):
        return False
    # small tolerance for float CSVs like 0.999999999
    return (abs(score - 100.0) < 1e-9) or (abs(score - 1.0) < 1e-9)

def copy_with_collision_handling(src: str, dest_dir: str) -> str:
    """
    Copy src into dest_dir, avoiding overwriting by appending (1), (2), ...
    Returns final destination path.
    """
    safe_makedirs(dest_dir)
    base = os.path.basename(src)
    name, ext = os.path.splitext(base)
    dst = os.path.join(dest_dir, base)

    k = 1
    while os.path.exists(dst):
        dst = os.path.join(dest_dir, f"{name} ({k}){ext}")
        k += 1

    shutil.copy2(src, dst)
    return dst

# -----------------------------
# 1) Read CSV and build ID -> has_any_100 boolean
#    If a company_id has ANY 100% match in the CSV, route ALL its PDFs to OUT_100.
# -----------------------------
df = pd.read_csv(CSV_PATH, dtype={ID_COL: "string"}, low_memory=False)

if ID_COL not in df.columns:
    raise ValueError(f"CSV missing required column: {ID_COL}")
if SCORE_COL not in df.columns:
    raise ValueError(f"CSV missing required column: {SCORE_COL} (edit SCORE_COL to your actual score column name)")

df[ID_COL] = df[ID_COL].apply(normalize_id)
df = df[df[ID_COL].notna()].copy()

df[SCORE_COL] = pd.to_numeric(df[SCORE_COL], errors="coerce")

# Boolean per row: is this row a 100% match?
df["_is_100"] = df[SCORE_COL].apply(is_hundred)

# For each company_id: does it have ANY 100% match row?
has_any_100_by_id = (
    df.groupby(ID_COL, dropna=True)["_is_100"]
      .any()
      .to_dict()
)

company_ids = list(has_any_100_by_id.keys())
print(f"Found {len(company_ids):,} unique company_id values.")

# -----------------------------
# 2) Index PDFs across batch folders by leading id
# -----------------------------
pdfs_by_id = {}  # id -> list of full paths

for batch_dir in BATCH_DIRS:
    if not os.path.isdir(batch_dir):
        print(f"WARNING: not a directory, skipping: {batch_dir}")
        continue

    for root, _, files in os.walk(batch_dir):
        for fn in files:
            if not fn.lower().endswith(".pdf"):
                continue
            leading = extract_leading_id(fn)
            if leading is None:
                continue

            full_path = os.path.join(root, fn)
            pdfs_by_id.setdefault(leading, []).append(full_path)

print(f"Indexed PDFs for {len(pdfs_by_id):,} distinct leading IDs across batch folders.")

# -----------------------------
# 3) Copy PDFs into folders based on "any 100% for that company_id"
# -----------------------------
safe_makedirs(OUT_100)
safe_makedirs(OUT_LT)

copied_100 = 0
copied_lt  = 0
missing_ids = 0

for cid, has_any_100 in has_any_100_by_id.items():
    matches = pdfs_by_id.get(cid, [])
    if not matches:
        missing_ids += 1
        continue

    out_base = OUT_100 if has_any_100 else OUT_LT

    # Put each company's files in its own subfolder (prevents filename collisions across companies)
    company_out_dir = os.path.join(out_base, cid)

    for src_pdf in matches:
        copy_with_collision_handling(src_pdf, company_out_dir)
        if has_any_100:
            copied_100 += 1
        else:
            copied_lt += 1

print("Done.")
print(f"Copied to 100% folder: {copied_100:,} PDFs")
print(f"Copied to <100% folder: {copied_lt:,} PDFs")
print(f"Company IDs with no matching PDFs found: {missing_ids:,}")

print("\nOutput folders:")
print(" -", OUT_100)
print(" -", OUT_LT)


Found 4,384 unique company_id values.
Indexed PDFs for 7,552 distinct leading IDs across batch folders.
Done.
Copied to 100% folder: 16,730 PDFs
Copied to <100% folder: 1,083 PDFs
Company IDs with no matching PDFs found: 0

Output folders:
 - D:\vc-research\vc-research\matched_pdfs_by_score\match_100
 - D:\vc-research\vc-research\matched_pdfs_by_score\match_less_than_100
